[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LizbethMG-Teaching/pose2behav-book/blob/main/notebooks/analysis_multi_animal.ipynb)

# 📓 Notebook 3 – Analysis of multi animal (top-view mouse)
## 1. Introduction & objectives

In this notebook, you will analyze pose estimation outputs generated with the SuperAnimal ModelZoo on a top view multi animal video containing several mice.

**Learning goals:**

After this notebook, you should be able to:
- Load and preprocess multi animal pose data from SuperAnimal DLC
- Implement your own filtering and interpolation choices
- Compute activity and social metrics per mouse
- Integrate the results into a summary table

---

**About this notebook**

In this notebook, you will analyze pose-estimation data from freely-moving mice.

# 🐭🐭🏠🎥 The Mouse House: multi animal pose challenge

<img src="https://raw.githubusercontent.com/LizbethMG-Teaching/pose2behav-book/main/assets/illustrations/cover-mice.png" width="50%">

Three mice stay together in the "Mouse House" 🐭🤍🏠, a fully monitored arena.
Every movement is tracked with SuperAnimal DeepLabCut.

Your task is to use pose data to build a behavioral profile for each mouse:
- Who is the Hyperactive One?
- Who is the Social Butterfly?
- Who is the Lone Wolf?
- And who wins each "medal" category?

🥇🥈🥉 At the end, you will assign gold, silver, and bronze medals in:
- Activity
- Sociability

For this exercise, you will work mostly independently, but everyone must create the same output variable names and structure so we can compare results.

---
**Instructions**

This notebook mixes pre-filled code cells (ready to run) and coding exercises that you will complete.

- Some cells are already complete (just run them).
- You are free to choose methods, but you must respect:
  - Input: the provided pose file
  - Output variable names and column names as indicated.

👉 Here’s how to work through it:
1. Read carefully each section before running the cells.
2. When a cell requires you to code, you’ll see a TODO comment.

⚡ After finishing the course, feel free to experiment and modify the notebook as you like!

---

<img src="https://raw.githubusercontent.com/LizbethMG-Teaching/pose2behav-book/main/assets/single-frame-3-animals.png" width="50%">

## Arena geometry and scale

**‼️ Useful reference information for this LAB**

- Frame resolution: 652 × 636 pixels
- Frame rate: 66 frames per second
- Mouse body length (nose to base of tail): 80 pixels
- Real body length: 9 cm
- Approximate scale: 1 cm ≈ 8.89 pixels
- Pixel to centimeter conversion: 1 pixel ≈ 0.1125 cm

Arena geometry

- Arena shape: circular
- Diameter in the image: 460 pixels
- Real-world diameter: about 52 cm
- For simplified calculations, the arena can be approximated as a 460 × 460 pixel square
- Upper left corner of this square: x = 108, y = −78 (image coordinate system)

---

## 2. Data Loading & Format Inspection

### 2.1 Download data (prefilled)

**📋 Instructions:**
- Run the code cell below to download the dataset file.

In [ ]:
# PREFILLED, NO NEED TO CHANGE, JUST RUN THIS CELL
# Install and import the required libraries:
!pip -q install gdown tables

import os
from pathlib import Path
import gdown, pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import groupby
import re
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle


# --------------------------------------------------------------

# Detect if running in Google Colab
if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
    DEST = Path("/content/cleaned_pose_downloaded.h5")
else:
    DEST = Path("cleaned_pose_downloaded.h5")  # save in current folder locally
print("Saving to:", DEST)

# Select here the experiment you want to download, comment the others:
# mice-5_5min

# File : "cleaned_pose_3mice_5min.h5"
# FILE_ID = "1BxdhvhPl-2Yb39v1_ZICi9fzp_wv8u8q"

# File : "cleaned_pose_3mice_4min.h5"
FILE_ID = "1x-aevTfg0PbwwUN2y8bFJ6l5_oNK5Chf"
URL = f"https://drive.google.com/uc?id={FILE_ID}"

print("Downloading from Drive...")
_ = gdown.download(URL, str(DEST), quiet=False)

# Basic checks
assert DEST.exists() and DEST.stat().st_size > 0, "❌ Download failed or empty file."
print(f"✅ Downloaded to {DEST} ({DEST.stat().st_size/1_000_000:.2f} MB)")

# --- Load the cleaned H5 file into a pandas DataFrame ---
df = pd.read_hdf(DEST, key="df_with_missing")

print("✅ Data loaded successfully!")
print("Shape:", df.shape)
print("Columns:", list(df.columns)[:8], "...")

### 2.1 Download data (prefilled)

👉🏼 Run the next cell to explore the cleaned file you loaded.

🤓 You have a little helper that you can use to get the data for a single mouse.

In [ ]:
# 👉🏼 PREFILLED CELL — JUST RUN IT

# Show the first few columns
print("\nFirst 10 columns:")
print(df.columns[:10])

# List animals present in this file
animals = sorted(set(col.split("_")[0] for col in df.columns))
print("\nAnimals in this file:", animals)

# Show how many columns each mouse has
print("\nNumber of columns per mouse:")
for a in animals:
    count = sum(col.startswith(a + "_") for col in df.columns)
    print(f"{a}: {count} columns")

# Function to extract a single mouse
def get_mouse(df, animal_id):
    """Returns a DataFrame with only one mouse's data and clean column names."""
    cols = [c for c in df.columns if c.startswith(f"{animal_id}_")]
    dfa = df[cols].copy()
    dfa.columns = [c.replace(f"{animal_id}_", "") for c in cols]
    return dfa

# Example: extract the first mouse
example_mouse = animals[0]
mouse_df = get_mouse(df, example_mouse)


In [ ]:
# Constants from the description
FRAME_RATE = 66  # frames per second
PIXEL_TO_CM = 0.1125

ARENA_DIAM_PX = 460
ARENA_CENTER_X = 108 + ARENA_DIAM_PX / 2
ARENA_CENTER_Y = -78 + ARENA_DIAM_PX / 2
ARENA_RADIUS_PX = ARENA_DIAM_PX / 2

print("Pixel to cm:", PIXEL_TO_CM)
print("Arena center (px):", ARENA_CENTER_X, ARENA_CENTER_Y)

## 3. From pixel coordinates to behavior features

So far you have:

- Loaded a **multi animal** pose DataFrame `df`
- Listed all animal IDs in the file in the list `animals`
- Learned how to extract a **single mouse** with `get_mouse(df, animal_id)`

### 3.1 Summary table for the multi animal challenge

Your goal in this notebook is to decide, for each mouse:

- How **active** it was  
- How **fast** it moved on average  
- How **social** it was  (percentage of time the animal was social)

To do this, we will build a summary table with one row per animal and the following columns:

- `id`  
- `total_distance_cm`  
- `mean_speed_cm_s`  
- `social_fraction`  

We will start by creating an empty table that contains only the animal IDs.  
In the next sections you will fill on your own the three metric columns step by step:

1. Distance traveled  
2. Mean speed  
3. Social fraction of time


In [ ]:
# Summary table that you will fill step by step

summary_df = pd.DataFrame({"id": animals})
summary_df["total_distance_cm"] = np.nan
summary_df["mean_speed_cm_s"] = np.nan
summary_df["social_fraction"] = np.nan

print("Initial summary table (only IDs are filled):")
display(summary_df)

## 4. Distance traveled by each animal

First, you will compute how far each mouse traveled in the arena.

For each mouse:

- Extract its trajectory using `get_mouse(df, animal_id)`  
- Use a reference body part (for example `mid_back`)  
- Compute the distance between consecutive frames  
- Convert from pixels to centimeters using `PIXEL_TO_CM`  
- Sum across the whole recording to obtain `total_distance_cm`  

The function below computes distance and mean speed for one mouse.  

Fill  only the `total_distance_cm` column of the summary table.

In [ ]:
# >>>>>>>>>>>>>>>>>>>
# TODO : Compute total distance traveled in cm for each animal, then
# fill the total_distance_cm column for all animals

# Add your code here: 👇🏼


# <<<<<<<<<<<<<<<<<<<


# Helper: kinematics for one mouse

def compute_kinematics_one_mouse(mouse_df, part="mid_back",
                                 frame_rate=FRAME_RATE,
                                 px_to_cm=PIXEL_TO_CM):
    """
    Compute total distance (cm) and mean speed (cm/s) for a single mouse.
    """
    "This grabs one body part's x and y coordinates across all frames"
    x = mouse_df[f"{part}_x"].to_numpy(float)
    y = mouse_df[f"{part}_y"].to_numpy(float)
    
    """valid[i] is:
	    True if both x and y exist for frame i
	    False otherwise
    """
    valid = np.isfinite(x) & np.isfinite(y)

    "Compute frame-to-frame displacement"
    dx = np.diff(x)
    dy = np.diff(y)
    
    """Imagine valid looks like this:
        Frame index:  0    1    2    3
        valid:       [T,   T,   F,   T]

        valid_step becomes:

        Step index:   0    1    2
        valid_step:  [T,   F,   F]
    """
    valid_step = valid[1:] & valid[:-1]

    "Compute step distances, set invalid steps to NaN"
    step_dist_px = np.sqrt(dx**2 + dy**2)
    step_dist_px[~valid_step] = np.nan

    "Convert from pixels → cm:"
    step_dist_cm = step_dist_px * px_to_cm
    "Total distance is the sum of all valid steps:"
    total_dist_cm = np.nansum(step_dist_cm)
    
    "Instantaneous speed is distance per frame times frames per second:"
    speed_cm_s = step_dist_cm * frame_rate
    mean_speed_cm_s = np.nanmean(speed_cm_s)

    return total_dist_cm, mean_speed_cm_s


# Use the helper to fill the total_distance_cm column for all animals

for a in animals:
    mouse_df = get_mouse(df, a)
    total_dist_cm, mean_speed_cm_s = compute_kinematics_one_mouse(mouse_df,
                                                                  part="mid_back")
    summary_df.loc[summary_df["id"] == a, "total_distance_cm"] = total_dist_cm

print("Summary table after filling distance traveled:")
display(summary_df)

### 4.1 Think about distance

Look at the `total_distance_cm` values in the summary table.

- Which mouse appears to be the most active according to total distance?  
- Are there large differences between animals or are they similar?  

If you want, you can reuse the plotting tools from the previous notebooks
to visualize the trajectories of the most and least active mice.

## 5. Mean speed for each animal

Distance tells you how far each mouse traveled.  
Now you will compute how fast they moved on average.

For each mouse:

- Reuse the `compute_kinematics_one_mouse` function  
- Extract the `mean_speed_cm_s` value  
- Fill the `mean_speed_cm_s` column of the summary table  

Note that two mice could travel similar distances but have different mean speeds,  
depending on how often they stopped or froze during the recording.

In [ ]:
# >>>>>>>>>>>>>>>>>>>
# TODO : Compute the average speed in cm/s per animal, then
# fill the mean_speed_cm_scolumn for all animals

# Add your code here: 👇🏼


# <<<<<<<<<<<<<<<<<<<

# Fill the mean_speed_cm_s column using the same helper

for a in animals:
    mouse_df = get_mouse(df, a)
    total_dist_cm, mean_speed_cm_s = compute_kinematics_one_mouse(mouse_df,
                                                                  part="mid_back")
    summary_df.loc[summary_df["id"] == a, "mean_speed_cm_s"] = mean_speed_cm_s

print("Summary table after filling distance and mean speed:")
display(summary_df)

### 5.1 Think about speed

Compare `total_distance_cm` and `mean_speed_cm_s` for all animals.

- Does the mouse with the largest distance also have the highest mean speed?  
- Is there a mouse that moves quickly but not very far, or slowly but over a long time?  
- How could freezing periods affect the mean speed values?

In the next section you will quantify how social each mouse is,  
and then you will have all three metrics needed for the challenge.

## 6. Social fraction of time

To quantify how social each mouse is, we define a simple sociality measure.

At a given frame, a mouse is in a **social state** if:

- Its reference body part is within a fixed distance of at least one other mouse  
- Both animals have valid detections at that frame  

We will use:

- a body part of your choice as the reference body part. Think about which bodypart would make sense.
- A social radius of `5 cm`  

For each mouse you will compute:

> `social_fraction` =  
> fraction of valid frames where the mouse is within 5 cm of at least one other mouse  

You will then fill the `social_fraction` column of the summary table.

In [ ]:
# >>>>>>>>>>>>>>>>>>>
# TODO : Compute the social fraction
# fill the social_fraction column for all animals

# Add your code here: 👇🏼


# <<<<<<<<<<<<<<<<<<<

# Helper: social fraction of time for all animals

def social_fraction_all_mice(df, animals, part="nose",
                             social_radius_cm=5.0,
                             px_to_cm=PIXEL_TO_CM):
    """
    Compute, for each mouse, the fraction of valid frames where it is
    within social_radius_cm of at least one other mouse.
    """
    coords = {}
    valid = {}
    for a in animals:
        x = df[f"{a}_{part}_x"].to_numpy(float)
        y = df[f"{a}_{part}_y"].to_numpy(float)
        v = np.isfinite(x) & np.isfinite(y)
        "Get its x and y coordinates across all frames"
        coords[a] = (x, y)
        "boolean array marking valid frames"
        valid[a] = v

        """Initialize a “social timeline” for each mouse
        Initially all False
    	Later, frames where the mouse is “social” will be set to True"""
    n_frames = len(df)
    social = {a: np.zeros(n_frames, dtype=bool) for a in animals}
    
    """
    We pick a “focal mouse” → ai
    Then compare it pairwise to all other mice aj.

    both_valid[i] is True only if:
	•	Mouse ai is valid at frame i
	•	Mouse aj is also valid at frame i

    If either is missing, we ignore that frame.
    """
    for ai in animals:
        xi, yi = coords[ai]
        vi = valid[ai]
        for aj in animals:
            if aj == ai:
                continue
            xj, yj = coords[aj]
            vj = valid[aj]
            both_valid = vi & vj
            """
            Compute distances between two mice across all frames, 
            This gives the full distance-time series for that pair.
            """            
            dx = xi - xj
            dy = yi - yj
            dist_cm = np.sqrt(dx**2 + dy**2) * px_to_cm
            """
            A frame is considered “social” if:
	        1.	The two mice are closer than the threshold (e.g. 5 cm)
	        2.	Both mice were tracked correctly at that frame
            close[i] is a boolean array marking these frames.
            """
            close = (dist_cm <= social_radius_cm) & both_valid
            
            """	
            If close[i] is True for ANY partner mouse aj,
            Then social[ai][i] becomes True, This is like saying:
            If Mouse 1 is close to Mouse 2 OR Mouse 3 → count the frame as social.
            """
            social[ai] |= close

    fractions = {}
    for a in animals:
        v = valid[a]
        if v.sum() == 0:
            fractions[a] = np.nan
        else:
            """Compute fraction of valid frames that are social, 
                it is computing a mean, but in this context a mean 
                of booleans is the same thing as a fraction."""
            fractions[a] = social[a][v].mean()
    return fractions


# Fill the social_fraction column of the summary table

social_frac = social_fraction_all_mice(df, animals,
                                       part="nose",
                                       social_radius_cm=5.0)

for a in animals:
    summary_df.loc[summary_df["id"] == a, "social_fraction"] = social_frac[a]

print("Summary table after filling all three metrics:")
display(summary_df)

### 6.1 Interpreting the summary table

You now have a complete summary table with one row per mouse and three metrics:

- `total_distance_cm`  
- `mean_speed_cm_s`  
- `social_fraction`  

Use this table to answer the following questions:

1. Which mouse is the most active?  
   Do you base your answer on distance, speed, or both?  
2. Which mouse is the lone wolf?  
   Look for the smallest social fraction.  
3. Which mouse is the most social?  
   Look for the largest social fraction.  

In the final step you will create a visualization that highlights  
the winners in each category using gold, silver and bronze bars.

## 7. Medal style visualization

To summarize your results in a compact and intuitive way,  
you will create a **medal style** visualization:

- For each metric  
  - Normalize scores between 0 and 1  
  - Rank the mice from best to worst  
  - Color the bars with:
    - Gold for the best (GOLD   = "#d4af37")  
    - Silver for second (SILVER = "#c0c0c0")
    - Bronze for third  (BRONZE = "#cd7f32")

The resulting figure will have three bar plots:

1. Total distance  
2. Mean speed  
3. Social fraction  

This will clearly show which mouse wins each category.

In [ ]:
# Medal style bar plots for the three metrics

# Medal style bar plots (NO normalization)

def medal_plot(summary_df):
    """
    Plot three bar charts using raw metric values (not normalized).

    Metrics taken from summary_df:
      - total_distance_cm
      - mean_speed_cm_s
      - social_fraction

    Bars are colored by rank:
      gold = highest value
      silver = second highest
      bronze = third highest
    """

    metric_info = [
        ("total_distance_cm", "Total distance (cm)"),
        ("mean_speed_cm_s", "Mean speed (cm/s)"),
        ("social_fraction", "Social fraction"),
    ]

    ids = summary_df["id"].tolist()
    n_animals = len(ids)

    # Medal colors
    GOLD   = "#d4af37"
    SILVER = "#c0c0c0"
    BRONZE = "#cd7f32"

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for ax, (col, title) in zip(axes, metric_info):
        values = summary_df[col].to_numpy(float)

        # Rank values highest to lowest
        order = np.argsort(-values)

        # Assign colors
        colors = np.array([BRONZE] * n_animals)
        if n_animals >= 1:
            colors[order[0]] = GOLD
        if n_animals >= 2:
            colors[order[1]] = SILVER

        # Plot raw values
        bars = ax.bar(ids, values, color=colors)
        ax.set_title(title)
        ax.set_ylabel("Raw value")

        # Labels above bars
        for i, v in enumerate(values):
            ax.text(i, v + (0.02 * max(values)), f"{v:.2f}",
                    ha="center", va="bottom", fontsize=8)

    fig.suptitle("Multi animal challenge: gold, silver and bronze per category")
    plt.tight_layout()
    plt.show()


# Run the medal plot
medal_plot(summary_df)

## 🎉🎉🎉 Congratulations if you made it this far! 🎉🎉🎉

You have now learned not only how to handle multi-animal pose estimation data, but also how much fun it can be to analyze it. In real experiments the situation can be a bit more complex, yet not necessarily more difficult. With pose data there is a wide range of analyses you can explore.

In this notebook you worked with just a few body parts, but you can always integrate more to build a richer picture of each animal’s behavior. You also computed single metrics, although as we discussed in the course, many other analyses are possible. Some examples are:

🐭 Heatmaps of spatial occupancy  
📊 Speed distributions  
🤝 More detailed social distance and interaction metrics over time  
📈 Co-activity plots to see whether two animals tend to move together  
🧩 More advanced approaches like clustering behavioral motifs  

What you choose to analyze always depends on your scientific question. Now that you know the fundamentals, feel free to explore your data creatively and try new ideas on your own.

Cheers, and happy analyzing! 🚀